In [2]:
# Final Preprocessing + Windowing Pipeline

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from joblib import dump


# ------------------------------------------
# FINAL APPROVED FEATURE LIST
# ------------------------------------------
FEATURES = [
    # Time features
    "hour", "dow", "is_weekend", "month", "season", "year", "is_holiday",

    # Weather features
    "AirTemperature(degC)", "HighestTemperature(degC)", "LowestTemperature(degC)",
    "RelativeHumidity(%)", "WindSpeed(m/s)", "WindDirection(deg)",
    "PrecipitationAmount(mm)", "MaximumPrecipitationIntensity(mm/h)",
    "AirPressure(hPa)", "wind_chill", "humidity_temp_interaction",
    "heating_season", "very_cold", "is_precip",

    # Lagged consumption
    "cons_lag_1", "cons_lag_2", "cons_lag_3",
    "cons_lag_24", "cons_lag_48", "cons_lag_72", "cons_lag_168",

    # Rolling means
    "cons_roll_mean_3", "cons_roll_mean_6", "cons_roll_mean_24",
    "cons_roll_mean_48", "cons_roll_mean_72", "cons_roll_mean_168",

    # Structural break
    "has_heat_pump",
]


# ------------------------------------------
# Function: Create windowed dataset
# ------------------------------------------
def create_windowed(X_df, y_series, lookback=24):
    X_list, y_list = [], []

    data = X_df.values
    target = y_series.values

    for i in range(len(X_df) - lookback):
        X_list.append(data[i : i + lookback])
        y_list.append(target[i + lookback])

    return np.array(X_list), np.array(y_list)


# ------------------------------------------
# MAIN PIPELINE
# ------------------------------------------
def main():
    print("Loading final dataset...")
    df = pd.read_csv("dataset_mansion_features_FINAL.csv", parse_dates=["MTime"])
    df = df.sort_values("MTime").reset_index(drop=True)

    # --------------------------------------
    # 1. TRAIN / TEST / EVAL SPLITS (BY DATE)
    # --------------------------------------

    train_mask = (df["MTime"] >= "2016-01-01") & (df["MTime"] <= "2020-09-30")
    test_mask  = (df["MTime"] >= "2020-10-01") & (df["MTime"] <= "2021-09-30")
    eval_mask  = (df["MTime"] >= "2021-10-01") & (df["MTime"] <= "2022-05-31")

    train_df = df.loc[train_mask].copy()
    test_df  = df.loc[test_mask].copy()
    eval_df  = df.loc[eval_mask].copy()

    print(f"Rows: Train={len(train_df)}, Test={len(test_df)}, Eval={len(eval_df)}")

    # --------------------------------------
    # 2. SCALE FEATURES (Fit on TRAIN ONLY)
    # --------------------------------------

    scaler = StandardScaler()

    train_df[FEATURES] = scaler.fit_transform(train_df[FEATURES])
    test_df[FEATURES]  = scaler.transform(test_df[FEATURES])
    eval_df[FEATURES]  = scaler.transform(eval_df[FEATURES])

    print("Scaling complete.")

    # --------------------------------------
    # 3. WINDOWING (24-hour lookback)
    # --------------------------------------

    lookback = 24

    X_train, y_train = create_windowed(train_df[FEATURES], train_df["Consumption"], lookback)
    X_test,  y_test  = create_windowed(test_df[FEATURES],  test_df["Consumption"],  lookback)
    X_eval,  y_eval  = create_windowed(eval_df[FEATURES],  eval_df["Consumption"],  lookback)

    print("Windowing complete!")
    print("X_train:", X_train.shape, "y_train:", y_train.shape)
    print("X_test :", X_test.shape, "y_test :", y_test.shape)
    print("X_eval :", X_eval.shape, "y_eval :", y_eval.shape)

    # --------------------------------------
    # 4. SAVE OUTPUTS
    # --------------------------------------

    np.save("X_train.npy", X_train)
    np.save("y_train.npy", y_train)
    np.save("X_test.npy",  X_test)
    np.save("y_test.npy",  y_test)
    np.save("X_eval.npy",  X_eval)
    np.save("y_eval.npy",  y_eval)

    dump(scaler, "feature_scaler.joblib")

    print("\nPreprocessing COMPLETE!")
    print("Saved: X_train.npy, y_train.npy, X_test.npy, y_test.npy, X_eval.npy, y_eval.npy, feature_scaler.joblib")


if __name__ == "__main__":
    main()


Loading final dataset...
Rows: Train=41451, Test=8737, Eval=5809
Scaling complete.
Windowing complete!
X_train: (41427, 24, 35) y_train: (41427,)
X_test : (8713, 24, 35) y_test : (8713,)
X_eval : (5785, 24, 35) y_eval : (5785,)

Preprocessing COMPLETE!
Saved: X_train.npy, y_train.npy, X_test.npy, y_test.npy, X_eval.npy, y_eval.npy, feature_scaler.joblib
